# GPIO Example 2 — Polling vs. EXTI

**Target:** STM32F103C8T6 · PB9 / EXTI9  
**Focus:** Detect a button transition by polling or an external interrupt.

## Feature classification

| Method | Response | CPU use | Choose when |
|---|---|---|---|
| Polling | At the next read | CPU checks continuously or periodically | Simple slow controls; easy debounce/state logic |
| EXTI interrupt | On selected edge | CPU can do other work between edges | Asynchronous event or low-latency notification |
| Edge | Rising / falling / both | Configurable per EXTI line | Match signal polarity and required transitions |

On STM32F1, EXTI0–EXTI15 are shared by pin number across ports: only one of PA0/PB0/PC0 can be selected as EXTI0 at a time. CubeMX configures the AFIO port selection and NVIC vector. The F103 has grouped IRQ vectors for some EXTI lines.

## Mode A — Polling

The repository button project polls PB9 but leaves it `GPIO_NOPULL`; first verify the board supplies an external bias. For this lesson, wire a normally-open button from PB9 to ground and enable a pull-up. Poll at a deliberate interval and debounce in application logic; a delay inside a critical interrupt is not a robust debounce strategy.

```c
if (HAL_GPIO_ReadPin(GPIOB, GPIO_PIN_9) == GPIO_PIN_RESET) {
    // Button is down; update state once per debounced press.
}
HAL_Delay(10);
```


## Mode B — EXTI on PB9

CubeMX: PB9 → GPIO_EXTI9, pull-up; choose falling edge to signal a press for this wiring; enable `EXTI9_5_IRQn`. Rising means released-to-high. The repository `STM32F103C8T6_Interrupt_push_buuton` selects rising edge and no pull, so its physical polarity depends on external wiring. The callback below only records the pin event; debounce and LED updates belong in the main loop.

```c
static volatile uint32_t button_events;

void HAL_GPIO_EXTI_Callback(uint16_t pin)
{
    if (pin == GPIO_PIN_9) {
        button_events++;
    }
}

// In the main loop, atomically take/clear the event count and debounce it.
uint32_t primask = __get_PRIMASK();
__disable_irq();
uint32_t pending_events = button_events;
button_events = 0U;
__set_PRIMASK(primask);
if (pending_events != 0U) {
    // Process event outside the ISR.
}
```


## Debounce classification

- **Polling debounce:** Require a stable sampled value for a chosen interval. Simple, but resolution depends on poll period.
- **Timer-based debounce:** On the first edge, mask/ignore further edges until a timer expires, then sample the stable level. Better for responsive main loops.
- **RC + Schmitt input:** Hardware filtering can help noisy/long wiring, but component values and input thresholds must be designed for the signal.

Do not use `HAL_Delay()` or toggle all LEDs in an EXTI callback. The repository EXTI project toggles LEDs directly from its callback; use that as a code-review exercise and replace the ISR work with the event-flag pattern above. EXTI reports transitions, not necessarily a debounced physical press.